# 04 — Exp 2 & Exp 3: PageMantle End-to-End & Online `DB-CSE` Skill Evolution

1. Enforce **Invariant #8** (`verify_split_isolation`): `pm_official_8` & `pm_eval_32` NEVER enter `pm_online_stream_24`.
2. Evaluate pre-evolution baselines (`B0 Direct`, `B1 Single Gate`, `B2 ProACT-Enterprise`, `DIAL-Skill-v0` on `Luna` & `Terra`).
3. Run 3 sequential online waves (`pm_online_stream_24`) comparing `Static-v0`, `Reflexion-Append`, `AWM-Workflow`, `DB-CSE w/o Escrow`, and `DB-CSE Full`.
4. Evaluate post-evolution generalization strictly on held-out `pm_official_8` + `pm_eval_32`.

In [ ]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "caches",
    "runs",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)


In [ ]:
# === CELL 1: Sync GitHub Repo (git clone / git pull) & Hot-Reload ===
REPO_URL = "https://github.com/<YOUR_USERNAME>/when2speak.git"  # Edit once with your repo URL
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    !pip install -q -e . --no-deps

%load_ext autoreload
%autoreload 2


In [ ]:
# === CELL 2: Bootstrap Secrets & Resolved Model Config (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import describe_secret_sources
from dial_skill.llm.preflight import load_resolved_models, resolve_model_for_role
from dial_skill.config import load_models_config, load_dial_skill_config, load_experiments_config

print("Secret lookup diagnostics:", json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT), indent=2))
MODELS_CFG = load_models_config()
DIAL_CFG = load_dial_skill_config()
EXP_CFG = load_experiments_config()
RESOLVED_MODELS = load_resolved_models(drive_root=DRIVE_ROOT)
print("Resolved models:", RESOLVED_MODELS.model_dump() if RESOLVED_MODELS else "Not yet generated (run Notebook 00)")


## Step 1: Load Splits, Enforce Split Isolation & Configure Mode

In [ ]:
from dial_skill.benchmarks.build_pagemantle_ext import verify_split_isolation
from dial_skill.benchmarks.pagemantle_env import PageMantleEnv
from dial_skill.eval.runners import estimate_run_cost_usd
from dial_skill.schemas import PageMantleTask
from dial_skill.utils.checkpoint import save_run_metadata

MODE = "pilot"  # "pilot" (subset) | "full" (all 64 tasks)
PM_DIR = DRIVE_ROOT / "datasets" / "pagemantle_ext"
RUN_MAIN = DRIVE_ROOT / "runs" / "main_pagemantle" / MODE
RUN_ONLINE = DRIVE_ROOT / "runs" / "online_dbcse_waves" / MODE

def _load_pm(name: str):
    p = PM_DIR / f"{name}.jsonl"
    return [PageMantleTask.model_validate_json(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]

pm_official = _load_pm("pm_official_8")
pm_stream = _load_pm("pm_online_stream_24")
pm_eval = _load_pm("pm_eval_32")
verify_split_isolation(pm_official, pm_stream, pm_eval, strict_counts=True)

eval_tasks = pm_official + (pm_eval[:8] if MODE == "pilot" else pm_eval)
stream_tasks = pm_stream[:8] if MODE == "pilot" else pm_stream
print(f"MODE={MODE} | Held-out eval tasks={len(eval_tasks)} | Online stream tasks={len(stream_tasks)}")
print("Estimated cost:", json.dumps(estimate_run_cost_usd(mode=MODE, include_pagemantle=True), indent=2))
save_run_metadata(RUN_MAIN, {"mode": MODE, "eval_count": len(eval_tasks), "stream_count": len(stream_tasks)})


## Step 2: Exp 2 — Pre-Evolution Evaluation on Held-Out PageMantle

In [ ]:
from dial_skill.baselines.direct_chat import DirectChatBaseline
from dial_skill.baselines.proact import ProACTBaseline
from dial_skill.baselines.single_gate import SingleGateBaseline
from dial_skill.core.pipeline import DialSkillAgent
from dial_skill.core.skill_store import SkillStore
from dial_skill.eval.judge import BlindedDualJudge
from dial_skill.eval.metrics import compute_pagemantle_metrics
from dial_skill.eval.runners import run_pagemantle_eval
from dial_skill.llm.jev_client import TriageClient
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.utils.cache import SQLiteAPICache

cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache.sqlite")
oai = OpenAIClient(models_cfg=MODELS_CFG, cache=cache)
luna_model = resolve_model_for_role("luna", MODELS_CFG, RESOLVED_MODELS)
terra_model = resolve_model_for_role("terra", MODELS_CFG, RESOLVED_MODELS)
backend = RESOLVED_MODELS.triage_backend if RESOLVED_MODELS else MODELS_CFG.roles.triage_backend
triage_client = TriageClient(models_cfg=MODELS_CFG, cache=cache, openai_client=oai, luna_model=luna_model, backend=backend)

env = PageMantleEnv(dataset_dir=DRIVE_ROOT / "datasets" / "pagemantle")
judge = BlindedDualJudge(openai_client=oai, primary_judge_model=terra_model)

pre_agents = {
    "B0_direct_luna": DirectChatBaseline(openai_client=oai, model=luna_model),
    "B0_direct_terra": DirectChatBaseline(openai_client=oai, model=terra_model),
    "B1_single_gate": SingleGateBaseline(openai_client=oai, gate_model=luna_model, executor_model=terra_model),
    "B2_proact_enterprise": ProACTBaseline(openai_client=oai, gate_model=luna_model, executor_model=terra_model),
    "DIAL_Skill_v0_luna": DialSkillAgent(
        triage_client=triage_client, openai_client=oai, gate_model=luna_model, executor_model=luna_model,
        skill_store=SkillStore.from_json_file(), cfg=DIAL_CFG, method_name="DIAL_Skill_v0_luna",
    ),
    "DIAL_Skill_v0_terra": DialSkillAgent(
        triage_client=triage_client, openai_client=oai, gate_model=luna_model, executor_model=terra_model,
        skill_store=SkillStore.from_json_file(), cfg=DIAL_CFG, method_name="DIAL_Skill_v0_terra",
    ),
}

pre_metrics = {}
for name, agent in pre_agents.items():
    eps = await run_pagemantle_eval(
        agent=agent, tasks=eval_tasks, env=env, output_jsonl=RUN_MAIN / f"preds_{name}.jsonl", resume=True
    )
    scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(eval_tasks, eps)]
    (RUN_MAIN / f"judge_{name}.jsonl").write_text("\n".join(s.model_dump_json() for s in scores) + "\n", encoding="utf-8")
    pre_metrics[name] = compute_pagemantle_metrics(eval_tasks, eps, scores)
    print(f"[{name}] Overall={pre_metrics[name]['overall_score']:.2f} | SilAcc={pre_metrics[name]['silence_accuracy']:.2f} | Aud={pre_metrics[name]['audience_and_location']:.2f}")

(RUN_MAIN / "pre_evolution_metrics.json").write_text(json.dumps(pre_metrics, indent=2), encoding="utf-8")


## Step 3: Exp 3 — Online Evolution (`DB-CSE Full` vs `DB-CSE w/o Escrow` vs `Reflexion` vs `AWM`)

In [ ]:
from dial_skill.eval.runners import run_online_evolution_stream
from dial_skill.evolution.baselines import AWMWorkflowUpdater, ReflexionAppendUpdater
from dial_skill.evolution.db_cse import DBCSEUpdater
from dial_skill.evolution.feedback import extract_feedback_from_pagemantle

# 1. DB-CSE Full (with Counterfactual Replay Escrow Gate & O(1) Cap)
dbcse_full_agent = DialSkillAgent(
    triage_client=triage_client, openai_client=oai, gate_model=luna_model, executor_model=terra_model,
    skill_store=SkillStore.from_json_file(), cfg=DIAL_CFG, method_name="DIAL_Skill_DBCSE_Full",
)
dbcse_full_updater = DBCSEUpdater(
    skill_store=dbcse_full_agent.skill_store, openai_client=oai, critic_model=terra_model,
    cfg=DIAL_CFG, log_path=RUN_ONLINE / "dbcse_full" / "skill_evolution_log.jsonl",
)
full_stream_summary = await run_online_evolution_stream(
    agent=dbcse_full_agent, updater=dbcse_full_updater, stream_tasks=stream_tasks,
    env=env, run_dir=RUN_ONLINE / "dbcse_full", judge=judge,
)

# 2. DB-CSE w/o Escrow Gate ablation
no_escrow_cfg = load_dial_skill_config(overrides={"evolution": {"enable_escrow_gate": False}})
dbcse_noesc_agent = DialSkillAgent(
    triage_client=triage_client, openai_client=oai, gate_model=luna_model, executor_model=terra_model,
    skill_store=SkillStore.from_json_file(), cfg=no_escrow_cfg, method_name="DIAL_Skill_DBCSE_NoEscrow",
)
dbcse_noesc_updater = DBCSEUpdater(
    skill_store=dbcse_noesc_agent.skill_store, openai_client=oai, critic_model=terra_model,
    cfg=no_escrow_cfg, log_path=RUN_ONLINE / "dbcse_no_escrow" / "skill_evolution_log.jsonl",
)
noesc_stream_summary = await run_online_evolution_stream(
    agent=dbcse_noesc_agent, updater=dbcse_noesc_updater, stream_tasks=stream_tasks,
    env=env, run_dir=RUN_ONLINE / "dbcse_no_escrow", judge=judge,
)

# 3. Evaluate evolved agents strictly on held-out eval_tasks
post_metrics = {}
for name, agent in [
    ("DIAL_Skill_Evolved_Full", dbcse_full_agent),
    ("DIAL_Skill_Evolved_NoEscrow", dbcse_noesc_agent),
]:
    eps = await run_pagemantle_eval(
        agent=agent, tasks=eval_tasks, env=env, output_jsonl=RUN_MAIN / f"preds_{name}.jsonl", resume=False
    )
    scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(eval_tasks, eps)]
    (RUN_MAIN / f"judge_{name}.jsonl").write_text("\n".join(s.model_dump_json() for s in scores) + "\n", encoding="utf-8")
    post_metrics[name] = compute_pagemantle_metrics(eval_tasks, eps, scores)
    print(f"[{name} Held-Out] Overall={post_metrics[name]['overall_score']:.2f} | SilAcc={post_metrics[name]['silence_accuracy']:.2f}")

(RUN_MAIN / "post_evolution_metrics.json").write_text(json.dumps(post_metrics, indent=2), encoding="utf-8")
